# 00 環境の確認とデータの下見

このノートブックに TODO はない。上から順に実行して、環境が動くことと、これから使うデータの形を確かめる（15分）。

前提:

- `make up` でコンテナが起動している
- `make data` で大きいデータを生成してある（まだなら、ターミナルで実行してから戻ってくる）

In [ ]:
from pyspark.sql import functions as F

from labkit import get_spark, large, large_path, list_files, load_seed_views, materialize, observe, timed

spark = get_spark("00_setup")

## 1. Spark UI を開く

上のセルに出た `Spark UI:` の URL をブラウザで開く。Spark UI は、このノートブックのカーネル（＝ SparkSession）が生きている間だけ見られる。

これから毎回見るタブ:

| タブ | 見るもの |
|---|---|
| Jobs | action を1回呼ぶたびに増えるジョブ。いつ計算が走ったかが分かる |
| Stages | ジョブを shuffle の境界で区切ったもの。タスク数、shuffle の読み書き量 |
| SQL / DataFrame | クエリごとの実行計画の図。各演算子を通った行数、読んだファイル数 |
| Storage | `cache()` したデータ |
| Environment | 今の設定値（`spark.sql.shuffle.partitions` など） |

## 2. データは2種類ある

| | 場所 | 行数 | 用途 |
|---|---|---|---|
| シード | `data/seed/*.csv` | 数百〜1.5万行 | MySQL に入っているものと同じ。SQL の問題と、結果の正しさを確かめる実験に使う |
| 大きいデータ | `data/large/`（Parquet） | 数百万〜1千万行 | 時間や shuffle の量を測る実験に使う |

テーブル構成はどちらも同じ。各テーブルの意味は `data/README.md` にある。

In [ ]:
load_seed_views(spark)
spark.sql('''
    SELECT 'customers' AS table_name, COUNT(*) AS n FROM customers
    UNION ALL SELECT 'products', COUNT(*) FROM products
    UNION ALL SELECT 'orders', COUNT(*) FROM orders
    UNION ALL SELECT 'order_items', COUNT(*) FROM order_items
    UNION ALL SELECT 'payments', COUNT(*) FROM payments
''').show()

In [ ]:
for name in ["customers", "products", "orders", "order_items", "payments", "orders_by_month"]:
    print(f"{large(spark, name).count():>12,} 行  ", end="")
    list_files(large_path(name))

In [ ]:
orders = large(spark, "orders")
orders.printSchema()
orders.show(5)

## 3. 大きいデータには偏りを仕込んである

実データでは「一部の大口顧客に注文が集中する」のが普通で、Spark の性能問題の多くはこの偏り（skew）から起きる。大きいデータでは、上位3顧客だけで注文の約16%を占めるようにしてある。

In [ ]:
total = orders.count()
(orders.groupBy("customer_id").count()
 .withColumn("share_pct", F.round(F.col("count") / total * 100, 1))
 .orderBy(F.desc("count")).show(5))

## 4. 計測の道具

各ノートブックで使う道具は `labkit/` に置いてある（中身は短いので、一度読んでおくと良い）。

- `timed("ラベル")`: かかった時間を表示する
- `materialize(df)`: 結果を捨てつつ、`df` を最後まで計算させる。時間を測るときに使う
- `observe(spark, "ラベル")`: その中で走ったジョブ・ステージ・ファイルの読み込みを表にする。Spark UI に出ている数字と同じもの

下のセルを実行して、出てきた表を Spark UI の Stages タブと見比べる。

In [ ]:
with observe(spark, "注文をステータス別に数える"):
    rows = orders.groupBy("status").count().collect()
print(*rows, sep="\n")

表の読み方:

- 1行が1ステージ。`tasks` はそのステージのタスク数（＝ 処理したパーティション数）
- `shuffle書き` があるステージは、次のステージへデータを配り直している
- `ファイル読み込み` は、実際に開いたファイル数・サイズ・読んだ行数

なぜステージが2つに分かれるのか、なぜ2つ目のタスク数がこの数なのかは、`03_shuffle` で扱う。

In [ ]:
spark.stop()